# RL Movie - ML-Agents Training Notebook

このノートブックは `Unity -> Build for Colab -> Colab 学習 -> Import Trained Model` の標準パイプライン用です。
`scenario_manifest.yaml` を読み、`run_id` 規約と実験メタデータを検証しながら学習を実行します。

---
## Step 0: 実験パラメータ

In [ ]:
#@title Basic Parameters { run: "auto" }
SCENARIO_NAME = "YourScenarioName"  #@param {type:"string"}
RUN_TAG = "baseline"  #@param {type:"string"}
RUN_ID = ""  #@param {type:"string"}
SPEC_VERSION = 1  #@param {type:"integer"}
HYPOTHESIS = "baseline environment and reward shaping"  #@param {type:"string"}
SEED = 1  #@param {type:"integer"}
BASELINE_RUN = ""  #@param {type:"string"}
MAX_STEPS = 500000  #@param {type:"integer"}
CHECKPOINT_INTERVAL = 50000  #@param {type:"integer"}
KEEP_CHECKPOINTS = 5  #@param {type:"integer"}
SUMMARY_FREQ = 10000  #@param {type:"integer"}
RESUME_TRAINING = False  #@param {type:"boolean"}

import os
import re
from datetime import datetime, timezone

def to_snake_case(name):
    return re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()

def build_run_id(scenario_slug, spec_version, run_tag):
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
    normalized_tag = re.sub(r"[^a-z0-9-]+", "-", run_tag.strip().lower()).strip("-")
    if not normalized_tag:
        raise ValueError("RUN_TAG must contain at least one lowercase letter or number")
    return f"{scenario_slug}__v{spec_version}__{normalized_tag}__{timestamp}"

SCENARIO_SLUG = to_snake_case(SCENARIO_NAME)
if not RUN_ID.strip():
    RUN_ID = build_run_id(SCENARIO_SLUG, SPEC_VERSION, RUN_TAG)

RUN_ID_PATTERN = rf"^{SCENARIO_SLUG}__v{SPEC_VERSION}__[a-z0-9]+(?:-[a-z0-9]+)*__\d{{8}}_\d{{4}}$"

DRIVE_PROJECT = "/content/drive/MyDrive/RL-Movie"
DRIVE_BUILDS = f"{DRIVE_PROJECT}/Builds"
DRIVE_RESULTS = f"{DRIVE_PROJECT}/Results"
DRIVE_MODELS = f"{DRIVE_PROJECT}/Models"
WORK_DIR = "/content/training"
BUILD_DIR = f"{WORK_DIR}/{SCENARIO_NAME}"
CONFIG_DIR = f"{BUILD_DIR}/Config"
RUN_RESULTS_DIR = f"{DRIVE_RESULTS}/{RUN_ID}"
MODEL_EXPORT_DIR = f"{DRIVE_MODELS}/{SCENARIO_NAME}/{RUN_ID}"
RECORDING_EXPORT_DIR = f"{MODEL_EXPORT_DIR}/recording_candidates"
RUN_CONTEXT_PATH = f"{RUN_RESULTS_DIR}/run_context.json"
RUN_PROGRESS_PATH = f"{RUN_RESULTS_DIR}/progress.json"
RUN_LOG_PATH = f"{RUN_RESULTS_DIR}/training_output.log"
UNITY_PLAYER_LOG_PATH = f"{RUN_RESULTS_DIR}/unity_player.log"
RECORDING_CANDIDATES_PATH = f"{RUN_RESULTS_DIR}/recording_candidates.json"

if CHECKPOINT_INTERVAL <= 0:
    raise ValueError('CHECKPOINT_INTERVAL must be greater than 0')
if KEEP_CHECKPOINTS <= 0:
    raise ValueError('KEEP_CHECKPOINTS must be greater than 0')
if SUMMARY_FREQ <= 0:
    raise ValueError('SUMMARY_FREQ must be greater than 0')

print(f"Scenario:      {SCENARIO_NAME}")
print(f"Scenario slug: {SCENARIO_SLUG}")
print(f"Run tag:       {RUN_TAG}")
print(f"Run ID:        {RUN_ID}")
print(f"Spec version:  {SPEC_VERSION}")
print(f"Hypothesis:    {HYPOTHESIS}")
print(f"Seed:          {SEED}")
print(f"Baseline run:  {BASELINE_RUN or '(none)'}")
print(f"Max steps:     {MAX_STEPS:,}")
print(f"Checkpoint:    every {CHECKPOINT_INTERVAL:,} steps (keep {KEEP_CHECKPOINTS})")
print(f"Summary freq:  {SUMMARY_FREQ:,}")
print(f"Resume:        {RESUME_TRAINING}")
print(f"Run pattern:   {RUN_ID_PATTERN}")

---
## Step 1: 依存関係と Google Drive

In [ ]:
#@title ML-Agents のインストール { run: "auto" }
import os
import subprocess
import sys
import textwrap

TRAINING_REQUIREMENTS = "mlagents==1.1.0\nprotobuf==3.20.3\nnumpy==1.23.5\ntorch==2.1.2\nonnx==1.15.0\nsetuptools<81"
TRAINING_REQUIREMENTS_PATH = "/content/rl_movie_training_requirements.txt"

with open(TRAINING_REQUIREMENTS_PATH, "w", encoding="utf-8") as stream:
    stream.write(TRAINING_REQUIREMENTS + "\n")

print("Pinned training requirements:")
print(TRAINING_REQUIREMENTS)

def is_supported_mlagents_python(version_info):
    return version_info.major == 3 and version_info.minor == 10 and 1 <= version_info.micro <= 12

TRAINING_ENV_KIND = "host-python"
TRAINING_PYTHON_BIN = sys.executable

if not is_supported_mlagents_python(sys.version_info):
    TRAINING_ENV_KIND = "micromamba-py310"
    TRAINING_ENV_DIR = "/content/.mlagents-py310"
    TRAINING_PYTHON_BIN = f"{TRAINING_ENV_DIR}/bin/python"
    MICROMAMBA_BIN = "/content/bin/micromamba"

    if not os.path.exists(TRAINING_PYTHON_BIN):
        print(f"Host Python {sys.version.split()[0]} is incompatible with mlagents 1.1.0. Creating a Python 3.10.12 training environment...")
        subprocess.check_call([
            "bash",
            "-lc",
            "set -euo pipefail; mkdir -p /content/bin; curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/bin bin/micromamba --strip-components=1"
        ])
        subprocess.check_call([MICROMAMBA_BIN, "create", "-y", "-p", TRAINING_ENV_DIR, "python=3.10.12", "pip"])
    else:
        print(f"Reusing Python 3.10 training environment: {TRAINING_ENV_DIR}")
else:
    print(f"Host Python {sys.version.split()[0]} is directly compatible with mlagents 1.1.0.")

subprocess.check_call([TRAINING_PYTHON_BIN, "-m", "pip", "install", "-q", "-r", TRAINING_REQUIREMENTS_PATH])

version_probe = textwrap.dedent(
    '''
    from importlib.metadata import PackageNotFoundError, version
    import sys
    import torch
    import onnx

    def safe_version(package_name):
        try:
            return version(package_name)
        except PackageNotFoundError:
            return "not-installed"

    print(f"Training Python: {sys.executable}")
    print(f"Training Python version: {sys.version.split()[0]}")
    print(f"ML-Agents installed: {safe_version('mlagents')}")
    print(f"ML-Agents envs installed: {safe_version('mlagents-envs')}")
    print(f"Protobuf installed: {safe_version('protobuf')}")
    print(f"Torch installed: {torch.__version__}")
    print(f"ONNX installed: {onnx.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    '''
)

probe = subprocess.run([TRAINING_PYTHON_BIN, "-c", version_probe], capture_output=True, text=True, check=True)
print(f"Training environment: {TRAINING_ENV_KIND}")
print(f"Requirements file: {TRAINING_REQUIREMENTS_PATH}")
print(probe.stdout)
if probe.stderr.strip():
    print(probe.stderr)


In [ ]:
#@title Google Drive のマウント { run: "auto" }
from google.colab import drive

drive.mount('/content/drive')

for directory in [DRIVE_BUILDS, DRIVE_RESULTS, DRIVE_MODELS]:
    os.makedirs(directory, exist_ok=True)

print(f"Builds:  {DRIVE_BUILDS}")
print(f"Results: {DRIVE_RESULTS}")
print(f"Models:  {DRIVE_MODELS}")

---
## Step 2: Unity ビルドの展開

In [ ]:
#@title ZIP を展開して構成を確認 { run: "auto" }
import glob
import shutil
import zipfile

zip_path = f"{DRIVE_BUILDS}/{SCENARIO_NAME}.zip"
if not os.path.exists(zip_path):
    available = glob.glob(f"{DRIVE_BUILDS}/*.zip")
    print(f"Missing build ZIP: {zip_path}")
    print("Available ZIP files:")
    for candidate in available:
        size_mb = os.path.getsize(candidate) / (1024 * 1024)
        print(f"  - {os.path.basename(candidate)} ({size_mb:.1f} MB)")
    raise FileNotFoundError(zip_path)

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall(WORK_DIR)

exec_files = glob.glob(f"{WORK_DIR}/**/*.x86_64", recursive=True)
for exec_file in exec_files:
    os.chmod(exec_file, 0o755)

config_files = glob.glob(f"{WORK_DIR}/**/Config/*.yaml", recursive=True)
config_dirs = sorted({os.path.dirname(path) for path in config_files})

if not config_dirs:
    raise FileNotFoundError(f'No Config directory found inside extracted build: {WORK_DIR}')

CONFIG_DIR = config_dirs[0]
BUILD_DIR = os.path.dirname(CONFIG_DIR)
manifest_path = os.path.join(CONFIG_DIR, 'scenario_manifest.yaml')

print(f"Build ZIP: {zip_path}")
print(f"Executables: {exec_files}")
print("Config files:")
for config_file in config_files:
    print(f"  - {config_file}")
print(f"Resolved build dir: {BUILD_DIR}")
print(f"Resolved config dir: {CONFIG_DIR}")
print(f"Manifest path: {manifest_path}")

if not exec_files:
    raise FileNotFoundError('No .x86_64 executable found in extracted build')

---
## Step 3: 契約検証と学習設定の確定

In [ ]:
#@title Validate Manifest and Run ID { run: "auto" }
import glob
import json
import subprocess
import yaml

if not HYPOTHESIS.strip():
    raise ValueError('HYPOTHESIS must not be empty')

if not re.fullmatch(RUN_ID_PATTERN, RUN_ID):
    raise ValueError(
        f"RUN_ID does not match the required pattern: {RUN_ID_PATTERN}\n"
        f"Current RUN_ID: {RUN_ID}"
    )

if not os.path.exists(manifest_path):
    raise FileNotFoundError(f"scenario_manifest.yaml not found: {manifest_path}")

with open(manifest_path, 'r', encoding='utf-8') as stream:
    manifest = yaml.safe_load(stream)

required_manifest_keys = [
    'scenario_name', 'scene_name', 'agent_class', 'behavior_name', 'learning_goal',
    'success_conditions', 'failure_conditions', 'observation_contract', 'action_contract',
    'reward_rules', 'randomization_knobs', 'difficulty_stages', 'visual_theme',
    'camera_plan', 'acceptance_criteria', 'baseline_run', 'spec_version'
]
missing_keys = [key for key in required_manifest_keys if key not in manifest]
if missing_keys:
    raise KeyError(f"manifest is missing required keys: {missing_keys}")

if manifest['scenario_name'] != SCENARIO_NAME:
    raise ValueError(f"SCENARIO_NAME ({SCENARIO_NAME}) does not match manifest scenario_name ({manifest['scenario_name']})")

if str(manifest['spec_version']) != str(SPEC_VERSION):
    raise ValueError(f"SPEC_VERSION ({SPEC_VERSION}) does not match manifest spec_version ({manifest['spec_version']})")

build_requirements_path = os.path.join(CONFIG_DIR, 'training_requirements.txt')
if os.path.exists(build_requirements_path):
    with open(build_requirements_path, 'r', encoding='utf-8') as stream:
        build_requirements = '\n'.join(
            line.strip()
            for line in stream.readlines()
            if line.strip() and not line.lstrip().startswith('#')
        )
    notebook_requirements = '\n'.join(
        line.strip()
        for line in TRAINING_REQUIREMENTS.splitlines()
        if line.strip() and not line.lstrip().startswith('#')
    )
    if build_requirements != notebook_requirements:
        raise ValueError(
            'Build ZIP training_requirements.txt does not match notebook pinned requirements.\n'
            f'Build file: {build_requirements_path}\n'
            f'Notebook file: {TRAINING_REQUIREMENTS_PATH}'
        )
    print(f"Training requirements verified: {build_requirements_path}")
else:
    raise FileNotFoundError(
        'Build ZIP is missing Config/training_requirements.txt.\n'
        'Rebuild the scenario with the latest `RLMovie > Build for Colab (Current Scene)` before training.\n'
        f'Missing file: {build_requirements_path}'
    )

exec_path = exec_files[0]
runtime_link_targets = [exec_path]
unity_player_path = os.path.join(BUILD_DIR, 'UnityPlayer.so')
if os.path.exists(unity_player_path):
    runtime_link_targets.append(unity_player_path)
runtime_link_targets.extend(
    sorted(glob.glob(os.path.join(BUILD_DIR, '**', 'Plugins', '*.so'), recursive=True))
)
runtime_link_targets = list(dict.fromkeys(runtime_link_targets))

def inspect_runtime_links(binary_path):
    probe = subprocess.run(['ldd', binary_path], capture_output=True, text=True, check=False)
    stdout = probe.stdout.strip()
    stderr = probe.stderr.strip()
    missing = [
        line.strip()
        for line in stdout.splitlines()
        if 'not found' in line
    ]
    return {
        'path': binary_path,
        'returncode': probe.returncode,
        'missing': missing,
        'stdout': stdout,
        'stderr': stderr,
    }

runtime_dependency_reports = []
missing_runtime_dependencies = []
for binary_path in runtime_link_targets:
    report = inspect_runtime_links(binary_path)
    runtime_dependency_reports.append(report)
    print(f"ldd check: {binary_path}")
    print(report['stdout'] or '[no stdout]')
    if report['stderr']:
        print(report['stderr'])
    if report['missing']:
        missing_runtime_dependencies.extend(
            f"{binary_path}: {line}"
            for line in report['missing']
        )

if missing_runtime_dependencies:
    raise RuntimeError(
        'Extracted Unity build has missing shared library dependencies.\n'
        'This usually means the Colab package was built with the wrong Linux target or is missing runtime files.\n\n'
        + '\n'.join(missing_runtime_dependencies)
    )

training_configs = [
    path for path in config_files
    if os.path.basename(path) not in {'scenario_manifest.yaml', 'training_requirements.txt'}
]
if not training_configs:
    raise FileNotFoundError('No training config YAML found next to scenario_manifest.yaml')

config_path = training_configs[0]
with open(config_path, 'r', encoding='utf-8') as stream:
    config = yaml.safe_load(stream)

behavior_names = list(config.get('behaviors', {}).keys())
if manifest['behavior_name'] not in behavior_names:
    raise ValueError(
        f"manifest behavior_name ({manifest['behavior_name']}) not found in training config behaviors ({behavior_names})"
    )

existing_run_dir = os.path.join(DRIVE_RESULTS, RUN_ID)
existing_checkpoint_files = sorted(glob.glob(f"{existing_run_dir}/**/*.pt", recursive=True))
existing_onnx_files = sorted(glob.glob(f"{existing_run_dir}/**/*.onnx", recursive=True))
existing_event_files = sorted(glob.glob(f"{existing_run_dir}/**/events.out.tfevents.*", recursive=True))

if RESUME_TRAINING:
    if not os.path.isdir(existing_run_dir):
        raise FileNotFoundError(
            'RESUME_TRAINING is enabled, but the run directory does not exist.\n'
            f'Missing directory: {existing_run_dir}\n'
            'Use the original RUN_ID from the run you want to continue, or disable RESUME_TRAINING for a fresh run.'
        )

    expected_resume_checkpoints = [
        os.path.join(existing_run_dir, behavior_name, 'checkpoint.pt')
        for behavior_name in behavior_names
    ]
    missing_resume_checkpoints = [
        path for path in expected_resume_checkpoints
        if not os.path.exists(path)
    ]

    if missing_resume_checkpoints:
        raise RuntimeError(
            'RESUME_TRAINING is enabled, but the run directory is not resumable.\n'
            'ML-Agents expects a checkpoint.pt file for each behavior before resume can work.\n\n'
            'Missing expected checkpoints:\n'
            + '\n'.join(missing_resume_checkpoints)
            + '\n\n'
            + f'Existing .pt files: {len(existing_checkpoint_files)}\n'
            + f'Existing .onnx files: {len(existing_onnx_files)}\n'
            + f'Existing event files: {len(existing_event_files)}\n\n'
            + 'This usually means a previous run exported early artifacts before failing, leaving a partial run directory. '
            + 'Use a new RUN_ID or disable RESUME_TRAINING to start clean.'
        )
else:
    if os.path.isdir(existing_run_dir):
        print(f"Fresh run mode: existing results directory will be overwritten by --force: {existing_run_dir}")
        print(f"Existing .pt files: {len(existing_checkpoint_files)}")
        print(f"Existing .onnx files: {len(existing_onnx_files)}")
        print(f"Existing event files: {len(existing_event_files)}")

for behavior_name, behavior_config in config.get('behaviors', {}).items():
    behavior_config['max_steps'] = MAX_STEPS
    behavior_config['checkpoint_interval'] = CHECKPOINT_INTERVAL
    behavior_config['keep_checkpoints'] = KEEP_CHECKPOINTS
    behavior_config['summary_freq'] = SUMMARY_FREQ
    print(f"{behavior_name}: max_steps -> {MAX_STEPS:,}")
    print(f"{behavior_name}: checkpoint_interval -> {CHECKPOINT_INTERVAL:,}")
    print(f"{behavior_name}: keep_checkpoints -> {KEEP_CHECKPOINTS}")
    print(f"{behavior_name}: summary_freq -> {SUMMARY_FREQ:,}")

with open(config_path, 'w', encoding='utf-8') as stream:
    yaml.dump(config, stream, sort_keys=False, default_flow_style=False, allow_unicode=True)

effective_baseline_run = BASELINE_RUN.strip() or str(manifest.get('baseline_run', '')).strip()
if BASELINE_RUN.strip() and str(manifest.get('baseline_run', '')).strip() and BASELINE_RUN.strip() != str(manifest.get('baseline_run', '')).strip():
    raise ValueError('BASELINE_RUN input does not match manifest baseline_run')

run_context = {
    'scenario_name': SCENARIO_NAME,
    'scenario_slug': SCENARIO_SLUG,
    'run_id': RUN_ID,
    'spec_version': SPEC_VERSION,
    'hypothesis': HYPOTHESIS,
    'seed': SEED,
    'baseline_run': effective_baseline_run,
    'max_steps': MAX_STEPS,
    'resume_training': RESUME_TRAINING,
    'checkpoint_interval': CHECKPOINT_INTERVAL,
    'keep_checkpoints': KEEP_CHECKPOINTS,
    'summary_freq': SUMMARY_FREQ,
    'zip_path': zip_path,
    'manifest_path': manifest_path,
    'config_path': config_path,
    'behavior_names': behavior_names,
    'runtime_dependency_reports': runtime_dependency_reports,
}

os.makedirs(RUN_RESULTS_DIR, exist_ok=True)
with open(RUN_CONTEXT_PATH, 'w', encoding='utf-8') as stream:
    json.dump(run_context, stream, ensure_ascii=False, indent=2)

print(json.dumps(run_context, ensure_ascii=False, indent=2))
print(f"Run context saved: {RUN_CONTEXT_PATH}")


In [ ]:
#@title Run Training { run: "auto" }
import glob
import html
import subprocess
from collections import deque
from IPython.display import HTML, display

exec_path = exec_files[0]
resume_flag = '--resume' if RESUME_TRAINING else '--force'

cmd = [
    TRAINING_PYTHON_BIN,
    '-m',
    'mlagents.trainers.learn',
    config_path,
    f'--env={exec_path}',
    f'--run-id={RUN_ID}',
    f'--seed={SEED}',
    '--no-graphics',
    f'--results-dir={DRIVE_RESULTS}',
    resume_flag,
]
cmd.extend(['--env-args', '-logFile', UNITY_PLAYER_LOG_PATH])

step_pattern = re.compile(r'Step:\s*([0-9]+)')
reward_pattern = re.compile(r'Mean Reward:\s*(-?[0-9]+(?:\.[0-9]+)?)')
elapsed_pattern = re.compile(r'Time Elapsed:\s*([0-9]+(?:\.[0-9]+)?)\s*s')
recent_logs = deque(maxlen=12)

def utc_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def collect_artifacts():
    run_dir = os.path.join(DRIVE_RESULTS, RUN_ID)
    checkpoints = glob.glob(f"{run_dir}/**/*.pt", recursive=True)
    onnx_models = glob.glob(f"{run_dir}/**/*.onnx", recursive=True)
    event_files = glob.glob(f"{run_dir}/**/events.out.tfevents.*", recursive=True)
    candidates = checkpoints + onnx_models + event_files
    latest_artifact = max(candidates, key=os.path.getmtime) if candidates else ''
    return {
        'checkpoint_count': len(checkpoints),
        'onnx_count': len(onnx_models),
        'event_file_count': len(event_files),
        'latest_artifact': os.path.basename(latest_artifact) if latest_artifact else '',
    }

progress = {
    'scenario_name': SCENARIO_NAME,
    'run_id': RUN_ID,
    'status': 'starting',
    'stage': 'training',
    'max_steps': MAX_STEPS,
    'current_step': 0,
    'mean_reward': None,
    'elapsed_seconds': 0.0,
    'checkpoint_count': 0,
    'onnx_count': 0,
    'event_file_count': 0,
    'latest_artifact': '',
    'recent_logs': [],
    'updated_utc': utc_now(),
}

def persist_progress():
    os.makedirs(RUN_RESULTS_DIR, exist_ok=True)
    progress['recent_logs'] = list(recent_logs)
    progress['updated_utc'] = utc_now()
    progress.update(collect_artifacts())
    with open(RUN_PROGRESS_PATH, 'w', encoding='utf-8') as stream:
        json.dump(progress, stream, ensure_ascii=False, indent=2)

def render_dashboard():
    percent = 0.0
    if MAX_STEPS:
        percent = min(progress['current_step'] / MAX_STEPS * 100.0, 100.0)

    reward_text = '-' if progress['mean_reward'] is None else f"{progress['mean_reward']:.3f}"
    logs_html = '<br>'.join(html.escape(line) for line in recent_logs) or 'Waiting for trainer output...'
    latest_artifact = progress['latest_artifact'] or '-'

    return f"""
    <div style='font-family:Arial,sans-serif;border:1px solid #d6dce5;border-radius:12px;padding:16px;background:#f7f9fc;'>
      <div style='display:flex;justify-content:space-between;align-items:center;gap:12px;flex-wrap:wrap;'>
        <div>
          <div style='font-size:20px;font-weight:700;color:#16324f;'>RL Movie Training Progress</div>
          <div style='font-size:13px;color:#52667a;'>Run ID: {html.escape(RUN_ID)}</div>
        </div>
        <div style='font-size:13px;color:#52667a;'>Status: <strong>{html.escape(progress['status'])}</strong></div>
      </div>
      <div style='margin-top:14px;height:14px;background:#dde6f0;border-radius:999px;overflow:hidden;'>
        <div style='width:{percent:.2f}%;height:100%;background:linear-gradient(90deg,#2e86de,#37c978);'></div>
      </div>
      <div style='margin-top:8px;font-size:12px;color:#52667a;'>{progress['current_step']:,} / {MAX_STEPS:,} steps ({percent:.1f}%)</div>
      <div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:10px;margin-top:14px;'>
        <div style='background:white;border-radius:10px;padding:10px;'><div style='font-size:12px;color:#6b7c8f;'>Mean reward</div><div style='font-size:24px;font-weight:700;color:#16324f;'>{reward_text}</div></div>
        <div style='background:white;border-radius:10px;padding:10px;'><div style='font-size:12px;color:#6b7c8f;'>Elapsed</div><div style='font-size:24px;font-weight:700;color:#16324f;'>{progress['elapsed_seconds']:.1f}s</div></div>
        <div style='background:white;border-radius:10px;padding:10px;'><div style='font-size:12px;color:#6b7c8f;'>Checkpoints</div><div style='font-size:24px;font-weight:700;color:#16324f;'>{progress['checkpoint_count']}</div></div>
        <div style='background:white;border-radius:10px;padding:10px;'><div style='font-size:12px;color:#6b7c8f;'>Event files</div><div style='font-size:24px;font-weight:700;color:#16324f;'>{progress['event_file_count']}</div></div>
      </div>
      <div style='margin-top:14px;background:white;border-radius:10px;padding:10px;'>
        <div style='font-size:12px;color:#6b7c8f;'>Latest artifact</div>
        <div style='font-size:14px;font-weight:600;color:#16324f;'>{html.escape(latest_artifact)}</div>
      </div>
      <div style='margin-top:14px;background:#0f1720;color:#d9e2ec;border-radius:10px;padding:12px;'>
        <div style='font-size:12px;color:#8ea3b7;margin-bottom:8px;'>Recent trainer logs</div>
        <div style='font-family:Consolas,Monaco,monospace;font-size:12px;line-height:1.5;'>{logs_html}</div>
      </div>
      <div style='margin-top:10px;font-size:12px;color:#52667a;'>progress.json: {html.escape(RUN_PROGRESS_PATH)}</div>
    </div>
    """

def tail_text_file(path, max_lines=40):
    if not os.path.exists(path):
        return f"[missing] {path}"
    with open(path, 'r', encoding='utf-8', errors='replace') as stream:
        lines = stream.readlines()
    tail = ''.join(lines[-max_lines:]).strip()
    return tail or f"[empty] {path}"

print('Training command:')
print(' '.join(cmd))

os.makedirs(RUN_RESULTS_DIR, exist_ok=True)
persist_progress()
display_handle = display(HTML(render_dashboard()), display_id=True)

with open(RUN_LOG_PATH, 'w', encoding='utf-8') as log_stream:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )

    for raw_line in process.stdout:
        line = raw_line.rstrip()
        if not line:
            continue

        log_stream.write(line + '\n')
        log_stream.flush()
        recent_logs.append(line)

        step_match = step_pattern.search(line)
        reward_match = reward_pattern.search(line)
        elapsed_match = elapsed_pattern.search(line)

        if step_match:
            progress['current_step'] = int(step_match.group(1))
        if reward_match:
            progress['mean_reward'] = float(reward_match.group(1))
        if elapsed_match:
            progress['elapsed_seconds'] = float(elapsed_match.group(1))

        progress['status'] = 'running'
        persist_progress()
        display_handle.update(HTML(render_dashboard()))

    return_code = process.wait()

progress['status'] = 'completed' if return_code == 0 else f"failed ({return_code})"
persist_progress()
display_handle.update(HTML(render_dashboard()))

if return_code != 0:
    trainer_log_tail = tail_text_file(RUN_LOG_PATH)
    unity_log_tail = tail_text_file(UNITY_PLAYER_LOG_PATH)
    print("Trainer log tail:")
    print(trainer_log_tail)
    print("\nUnity player log tail:")
    print(unity_log_tail)
    raise RuntimeError(
        f"Training failed with exit code {return_code}\n\n"
        f"Trainer log: {RUN_LOG_PATH}\n"
        f"Unity player log: {UNITY_PLAYER_LOG_PATH}\n\n"
        f"===== Trainer log tail =====\n{trainer_log_tail}\n\n"
        f"===== Unity player log tail =====\n{unity_log_tail}"
    )

print(f"Training log saved: {RUN_LOG_PATH}")


---
## Step 4: 結果確認と成果物の保存

In [ ]:
#@title View TensorBoard { run: "auto" }
%load_ext tensorboard
%tensorboard --logdir {RUN_RESULTS_DIR}

In [ ]:
#@title Export Model and Summary { run: "auto" }
import glob
import shutil

run_results_root = os.path.join(DRIVE_RESULTS, RUN_ID)
models = sorted(glob.glob(f"{run_results_root}/**/*.onnx", recursive=True))
checkpoints = sorted(glob.glob(f"{run_results_root}/**/*.pt", recursive=True))
copied_models = []
checkpoint_candidates = []
pt_by_stem = {os.path.splitext(path)[0]: path for path in checkpoints}

os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)
os.makedirs(RECORDING_EXPORT_DIR, exist_ok=True)
os.makedirs(RUN_RESULTS_DIR, exist_ok=True)

for model_path in models:
    model_name = os.path.basename(model_path)
    model_stem = os.path.splitext(model_name)[0]
    checkpoint_match = re.match(r'^(?P<behavior>.+)-(?P<step>\d+)$', model_stem)

    if checkpoint_match:
        behavior_name = checkpoint_match.group('behavior')
        step = int(checkpoint_match.group('step'))
        checkpoint_dir = os.path.join(RECORDING_EXPORT_DIR, behavior_name, f"step_{step:07d}")
        os.makedirs(checkpoint_dir, exist_ok=True)

        model_destination = os.path.join(checkpoint_dir, model_name)
        shutil.copy2(model_path, model_destination)
        copied_models.append(model_destination)

        candidate = {
            'behavior_name': behavior_name,
            'step': step,
            'onnx_source_path': model_path,
            'onnx_path': model_destination,
        }

        checkpoint_path = pt_by_stem.get(os.path.splitext(model_path)[0])
        if checkpoint_path:
            checkpoint_destination = os.path.join(checkpoint_dir, os.path.basename(checkpoint_path))
            shutil.copy2(checkpoint_path, checkpoint_destination)
            candidate['checkpoint_source_path'] = checkpoint_path
            candidate['checkpoint_path'] = checkpoint_destination

        checkpoint_candidates.append(candidate)
        continue

    destination = os.path.join(MODEL_EXPORT_DIR, model_name)
    shutil.copy2(model_path, destination)
    copied_models.append(destination)

checkpoint_candidates.sort(key=lambda item: (item['behavior_name'], item['step']))
recording_candidates = {
    'scenario_name': SCENARIO_NAME,
    'run_id': RUN_ID,
    'checkpoint_interval': CHECKPOINT_INTERVAL,
    'keep_checkpoints': KEEP_CHECKPOINTS,
    'recording_export_dir': RECORDING_EXPORT_DIR,
    'checkpoint_count': len(checkpoint_candidates),
    'latest_checkpoint': checkpoint_candidates[-1] if checkpoint_candidates else None,
    'checkpoints': checkpoint_candidates,
}

with open(RECORDING_CANDIDATES_PATH, 'w', encoding='utf-8') as stream:
    json.dump(recording_candidates, stream, ensure_ascii=False, indent=2)

run_summary = {
    'scenario_name': SCENARIO_NAME,
    'scenario_slug': SCENARIO_SLUG,
    'run_id': RUN_ID,
    'spec_version': SPEC_VERSION,
    'hypothesis': HYPOTHESIS,
    'seed': SEED,
    'baseline_run': run_context['baseline_run'],
    'resume_training': RESUME_TRAINING,
    'max_steps': MAX_STEPS,
    'checkpoint_interval': CHECKPOINT_INTERVAL,
    'keep_checkpoints': KEEP_CHECKPOINTS,
    'summary_freq': SUMMARY_FREQ,
    'timestamp_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'zip_path': zip_path,
    'config_path': config_path,
    'manifest_path': manifest_path,
    'behavior_names': behavior_names,
    'model_export_dir': MODEL_EXPORT_DIR,
    'run_context_path': RUN_CONTEXT_PATH,
    'progress_path': RUN_PROGRESS_PATH,
    'training_log_path': RUN_LOG_PATH,
    'unity_player_log_path': UNITY_PLAYER_LOG_PATH,
    'copied_models': copied_models,
    'recording_candidates_path': RECORDING_CANDIDATES_PATH,
    'checkpoint_candidates': checkpoint_candidates,
    'manifest': manifest,
}

summary_path = os.path.join(RUN_RESULTS_DIR, 'run_summary.json')
with open(summary_path, 'w', encoding='utf-8') as stream:
    json.dump(run_summary, stream, ensure_ascii=False, indent=2)

print(f"Models copied to: {MODEL_EXPORT_DIR}")
for model_path in copied_models:
    print(f"  - {model_path}")
print(f"Recording candidates: {RECORDING_CANDIDATES_PATH}")
print(f"Checkpoint candidates found: {len(checkpoint_candidates)}")
print(f"Progress file: {RUN_PROGRESS_PATH}")
print(f"Training log: {RUN_LOG_PATH}")
print(f"Unity player log: {UNITY_PLAYER_LOG_PATH}")
print(f"Run summary: {summary_path}")

if not copied_models:
    print('Warning: no ONNX models were found in the results directory.')

In [ ]:
#@title 最新モデルをダウンロードする (任意) { run: "auto" }
from google.colab import files

download_candidate = ''
if checkpoint_candidates:
    download_candidate = checkpoint_candidates[-1]['onnx_path']
elif copied_models:
    download_candidate = copied_models[-1]

if download_candidate:
    print(f"Downloading: {os.path.basename(download_candidate)}")
    files.download(download_candidate)
else:
    print('No model available for download.')